# Deep learning 

In [6]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


cement = pd.read_csv("../Datasets/cement data.csv")
construction = pd.read_csv("../Datasets/construction_project_performance_dataset.csv")
supply = pd.read_csv("../Datasets/supply_chain.csv")


supply['Date'] = pd.to_datetime(supply['Date'], errors='coerce')
cement['Month'] = pd.to_datetime(cement['Month'], format='%b-%y', errors='coerce')


supply['YearMonth'] = supply['Date'].dt.to_period('M')
cement['YearMonth'] = cement['Month'].dt.to_period('M')

merged_data = pd.merge(supply, cement, on='YearMonth', how='left')


merged_data = merged_data.drop(columns=['Date', 'Month', 'YearMonth'], errors='ignore')


n = len(merged_data)
num_repeats = (n // len(construction)) + 1
construction_expanded = pd.concat([construction] * num_repeats, ignore_index=True).iloc[:n]

combined_data = pd.concat([merged_data.reset_index(drop=True),
                           construction_expanded.reset_index(drop=True)], axis=1)

print("After combining:", combined_data.shape)


combined_data = combined_data.fillna(0)


le = LabelEncoder()

for col in combined_data.select_dtypes(include=['object']).columns:
    combined_data[col] = le.fit_transform(combined_data[col].astype(str))


scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(combined_data)


def create_sequences(data, seq_length=5):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length][-1])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, seq_length=5)

print("Input Shape:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

y_train
# model = Sequential()

# model.add(LSTM(64, return_sequences=True, input_shape=(X.shape[1], X.shape[2])))
# model.add(Dropout(0.2))

# model.add(LSTM(32))
# model.add(Dropout(0.2))

# model.add(Dense(1))

# model.compile(optimizer='adam', loss='mse')

# model.summary()

# history = model.fit(
#     X_train, y_train,
#     epochs=20,
#     batch_size=16,
#     validation_data=(X_test, y_test)
# )

# loss = model.evaluate(X_test, y_test)
# print("Test Loss:", loss)

# predictions = model.predict(X_test)
# print("Sample Predictions:", predictions[:5])


After combining: (91250, 42)
Input Shape: (91245, 5, 42)


array([0., 0., 0., ..., 0., 0., 0.], shape=(72996,))